# 🔍 K-Nearest Neighbors (k-NN) na datech diabetu
**Kurz: Data Science / Machine Learning**  
**Sekce: Homework mezi dny 2 a 3 (Klasifikace)**

### Cíl cvičení:
1. Načíst předzpracovaný a škálovaný dataset `diabetes_scaled.csv`.
2. Rozdělit data na trénovací a testovací sadu v poměru 70/30 se zachováním stratifikace tříd.
3. Definovat mřížku hyperparametrů pro k-NN (`n_neighbors`, `metric`).
4. Použít `GridSearchCV` k nalezení optimální kombinace hyperparametrů a zvážit volbu optimalizační metriky (Accuracy vs. Recall vs. F1).
5. Natrénovat model s nejlepšími parametry z mřížky.
6. Otestovat efektivitu klasifikátoru na testovací sadě pomocí zvolených metrik a matice záměn.


In [ ]:
# Krok 1: Import knihoven
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report, roc_curve
)

sns.set_theme(style='whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)


## Krok 2: Načtení škálovaných dat
Načteme soubor `diabetes_scaled.csv`, který byl standardizován v předchozím kroku.

In [ ]:
# Načtení datasetu
df = pd.read_csv('data/diabetes_scaled.csv')
print(f"Rozměry datasetu: {df.shape[0]} řádků, {df.shape[1]} sloupců")

features = [c for c in df.columns if c != 'outcome']
X = df[features].values
y = df['outcome'].values

df.head()


## Krok 3: Rozdělení na trénovací a testovací data (70/30)
Použijeme poměr 70/30 a argument `stratify=y`, aby byl poměr diabetiček v obou sadách zachován.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)

print(f"Trénovací sada: {X_train.shape[0]} pacientek (Diabetiček: {sum(y_train == 1)})")
print(f"Testovací sada:  {X_test.shape[0]} pacientek (Diabetiček: {sum(y_test == 1)})")


## Krok 4 & 5: Definice hyperparametrů a inicializace modelu
Vytvoříme instanci `KNeighborsClassifier` a definujeme mřížku pro počet sousedů `n_neighbors` a metriku vzdálenosti `metric`.

In [ ]:
# Mřížka hyperparametrů
param_grid = {
    'n_neighbors': list(range(1, 31)),
    'metric': ['euclidean', 'manhattan', 'minkowski']
}

# Instance modelu
knn = KNeighborsClassifier()


## Krok 6: Hledání optimálních hyperparametrů pomocí GridSearchCV
**Úvaha k volbě metriky:**
V medicínské diagnostice diabetu je zásadní minimalizovat falešně negativní případy (nepoznaný diabetes).
Proto volíme jako optimalizační metriku `f1` (harmonický průměr přesnosti a záchytu) nebo `recall`.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid_search = GridSearchCV(
    estimator=knn,
    param_grid=param_grid,
    scoring='f1',  # Vyvážená metrika preferující záchyt
    cv=cv,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print(f"Nejlepší F1 skóre na validační křížové sadě: {grid_search.best_score_:.4f}")
print(f"Optimální hyperparametry: {grid_search.best_params_}")


## Krok 7 & 8: Trénování a testování na testovací sadě
Použijeme model s nejlepšími parametry vrácenými z `GridSearchCV`.

In [ ]:
best_knn = grid_search.best_estimator_

# Predikce na testovací sadě
y_pred = best_knn.predict(X_test)
y_prob = best_knn.predict_proba(X_test)[:, 1]

# Výpočet klíčových metrik
acc = accuracy_score(y_test, y_pred)
rec = recall_score(y_test, y_pred)
prec = precision_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
auc = roc_auc_score(y_test, y_prob)

print("=" * 60)
print("EVALUACE NA TESTOVACÍ SADĚ (70/30)")
print("=" * 60)
print(f"Accuracy (Přesnost):  {acc * 100:.2f} %")
print(f"Recall (Záchyt):      {rec * 100:.2f} %")
print(f"Precision (Preciznost): {prec * 100:.2f} %")
print(f"F1-Score:             {f1:.4f}")
print(f"ROC-AUC:              {auc:.4f}")
print("=" * 60)
print("\nKlasifikační zpráva:")
print(classification_report(y_test, y_pred, target_names=['Zdravá (0)', 'Diabetes (1)']))


## Vizualizace: Matice záměn (Confusion Matrix)

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(7, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Predikce: Zdravá (0)', 'Predikce: Diabetes (1)'],
            yticklabels=['Skutečnost: Zdravá (0)', 'Skutečnost: Diabetes (1)'])
plt.title(f'Matice záměn k-NN (k={best_knn.n_neighbors}, metrika={best_knn.metric})', fontsize=13)
plt.tight_layout()
plt.show()
